# Síntesis posterior al EDA y selección de datos para plata

Análisis exploratorio independiente de bronce. Fecha de revisión: 13 de noviembre de 2026.

Se examinan los archivos locales originales, no una tabla Delta ejecutada en Databricks.
No se limpia ni sobrescribe la fuente. Las transformaciones que aparezcan se hacen en copias
para evaluar lo que necesitará plata. Los resultados no prueban causalidad ni certifican seguridad.

Cada notebook sigue el orden: procedencia y unidad, calidad por columna y geometría,
distribuciones, límites y reglas para plata. No se muestran registros individuales de edificios oficiales.

In [ ]:
from pathlib import Path
import sys, json, re, unicodedata
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "CLAUDE.md").exists() and (p / "data/externos").is_dir())
sys.path.insert(0, str(ROOT / "analisis_bronce"))
from soporte import (ficha, huella, leer_geojson, mostrar_perfil, geometria, conteos,
                     faltantes, guardar_resumen, grafico_barras, SALIDAS, CRS_METROS)
pd.set_option("display.max_columns", 24)
pd.set_option("display.max_rows", 60)
plt.rcParams.update({"font.size": 11, "figure.dpi": 110})

## 1. Criterios declarados para elegir fuentes

Se aplica la pregunta de la guía: afectación en Cali y medidas que reduzcan el impacto
sobre población, infraestructura y servicios. La guía recomienda uno o dos cruces sólidos.

1. Relevancia para una acción verificable, sin confundir predicción, observación y simulación.
2. Datos materializados, trazables y con llaves de unión.
3. Cobertura geográfica/temporal y faltantes medidos, sin extrapolar la muestra al universo.
4. Cruce que no multiplique filas y unidades/denominadores coherentes.
5. Capacidad de ejecución con los recursos del evento y claridad para explicar limitaciones.

Primero se revisan los 17 notebooks independientes. Esta síntesis solo se ejecuta después
y añade comprobaciones de cruce. Decide el alcance de la siguiente plata; no modifica sus tablas,
los pesos ni los criterios de riesgo del pipeline existente. Sí se corrigió la pérdida de precisión WKT descubierta en la carga; el cambio está probado por regresión.

In [ ]:
from importlib.metadata import version
individuales=sorted((ROOT/"analisis_bronce/notebooks").glob("[0-1][0-9]_*.ipynb"))
perfiles={}
for p in individuales:
 path=SALIDAS/(p.stem+".json")
 if not path.exists():raise FileNotFoundError(f"Ejecutar primero el notebook independiente: {p.name}")
 perfiles[p.stem]=json.loads(path.read_text())
 # Cada resumen debe corresponder todavía a los mismos bytes crudos.
 assert huella(ROOT/perfiles[p.stem]["archivo"])==perfiles[p.stem]["sha256"],p.stem
inventario=pd.DataFrame([{"dataset":k,"filas":v.get("filas"),"archivo":v["archivo"],
                         "sha256":v["sha256"][:16],"perfil_disponible":True} for k,v in perfiles.items()])
display(inventario)
display(pd.Series({p:version(p) for p in ["pandas","geopandas","shapely","pyproj","scipy","matplotlib"]}).to_frame("version"))
assert len(perfiles)==17

## 2. Cobertura territorial y cardinalidad de los cruces

Se usa el centroide proyectado de cada contorno reparado en una copia. Se conserva el código
original para medir discrepancias. En plata deben distinguirse no coincidencia, coincidencia única
y coincidencia múltiple, en vez de tomar siempre la primera fila.

In [ ]:
raw,predios,ed=leer_geojson(ROOT/"data/oficiales/Datosprediales.json")
com=gpd.read_file(ROOT/"data/externos/osm_comunas.geojson").to_crs(CRS_METROS)
bar=gpd.read_file(ROOT/"data/externos/osm_barrios.geojson").to_crs(CRS_METROS)
com["comuna_codigo"]=com["name"].str.extract(r"(\d+)")[0].astype(int)
fixed=ed.geometry.make_valid()
metric=gpd.GeoDataFrame(geometry=gpd.GeoSeries(fixed,crs=ed.crs)).to_crs(CRS_METROS)
pts=gpd.GeoDataFrame(geometry=metric.centroid,crs=CRS_METROS)
j=gpd.sjoin(pts,com[["comuna_codigo","geometry"]],how="left",predicate="intersects")
n_coincidencias=j.groupby(level=0).index_right.count().reindex(pts.index,fill_value=0)
codigo_asignado=j.groupby(level=0).comuna_codigo.first().reindex(pts.index)
original=pd.to_numeric(predios.COMUNA.astype("string").str.strip(),errors="coerce")
unica=n_coincidencias.eq(1)
discrepa=unica & original.notna() & original.ne(codigo_asignado)
union_com=com.geometry.union_all();union_bar=bar.geometry.union_all()
territorio={"predios":len(predios),"sin_comuna_espacial":int(n_coincidencias.eq(0).sum()),
 "con_una_comuna":int(unica.sum()),"con_multiples_comunas":int(n_coincidencias.gt(1).sum()),
 "comuna_original_faltante":int(original.isna().sum()),"original_vs_osm_discrepancias":int(discrepa.sum()),
 "area_comunas_no_cubierta_por_barrios_km2":float(union_com.difference(union_bar).area/1e6),
 "area_barrios_fuera_comunas_km2":float(union_bar.difference(union_com).area/1e6)}
bj=gpd.sjoin(pts,bar[["osm_id","geometry"]],how="left",predicate="intersects")
nb=bj.groupby(level=0).index_right.count().reindex(pts.index,fill_value=0)
territorio.update({"predios_sin_barrio_osm":int(nb.eq(0).sum()),"predios_multiples_barrios_osm":int(nb.gt(1).sum())})
display(pd.Series(territorio).to_frame("resultado"))
assert n_coincidencias.eq(0).sum()+unica.sum()+n_coincidencias.gt(1).sum()==len(predios)

## 3. Salud: qué cambia al distinguir urgencias declaradas

Son distancias euclidianas de centroide a punto OSM. No representan tiempos de viaje ni capacidad.
No se descartan sitios cercanos fuera de una comuna: los servicios pueden atender a través de sus límites.

In [ ]:
salud=gpd.read_file(ROOT/"data/externos/osm_salud.geojson").to_crs(CRS_METROS)
urg=salud.loc[salud["emergency"].eq("yes")]
def distancias(destinos):
 join=gpd.sjoin_nearest(pts,destinos[["geometry"]],how="left",distance_col="distancia_m")
 return join.groupby(level=0).distancia_m.min().reindex(pts.index)
da=distancias(salud);du=distancias(urg)
dist=pd.DataFrame({"cualquier_sitio_salud_m":da,"urgencias_declaradas_m":du})
display(dist.describe(percentiles=[.5,.9,.95]))
assert du.notna().all() and (du>=da-1e-6).all()
# Resumen agregado: no se exportan distancias por predio.
salud_stats={"sitios_total":len(salud),"sitios_con_urgencias_declaradas":len(urg),
 "mediana_distancia_cualquier_sitio_m":float(da.median()),
 "mediana_distancia_urgencias_declaradas_m":float(du.median()),
 "p90_distancia_cualquier_sitio_m":float(da.quantile(.9)),
 "p90_distancia_urgencias_declaradas_m":float(du.quantile(.9))}
display(pd.Series(salud_stats).to_frame("valor"))
bins_comunes=np.linspace(0,max(da.max(),du.max()),31)
fig,ax=plt.subplots(figsize=(9,4));ax.hist(da,bins=bins_comunes,alpha=.6,label="Cualquier sitio de salud");ax.hist(du,bins=bins_comunes,alpha=.6,label="Urgencias declaradas")
ax.set(title="La definición de servicio cambia la distancia estimada",xlabel="Metros euclidianos desde centroides de la muestra",ylabel="Predios")
ax.legend();fig.tight_layout();fig.savefig(SALIDAS/"99_distancias_salud.png",dpi=150);plt.show()

## 4. Coherencia de la grilla de sitio y sus funciones

In [ ]:
grid=pd.read_csv(ROOT/"data/externos/gem_microzonas_grilla.csv",dtype={"ampcode":str})
af=pd.read_csv(ROOT/"data/externos/gem_amplificacion_suelo.csv",comment="#",dtype={"ampcode":str})
grid_geo=gpd.GeoDataFrame(grid,geometry=gpd.points_from_xy(grid.lon,grid.lat),crs=4326).to_crs(CRS_METROS)
near=gpd.sjoin_nearest(pts,grid_geo[["ampcode","geometry"]],how="left",distance_col="distancia_grilla_m")
dist_grid=near.groupby(level=0).distancia_grilla_m.min().reindex(pts.index)
codes=near.groupby(level=0).ampcode.first().reindex(pts.index)
no_funcion=set(grid.ampcode)-set(af.ampcode)
soporte={"codigos_grilla_sin_funcion":sorted(no_funcion),
 "distancia_grilla_mediana_m":float(dist_grid.median()),"distancia_grilla_p95_m":float(dist_grid.quantile(.95)),
 "distancia_grilla_maxima_m":float(dist_grid.max()),"predios_con_mas_de_un_vecino_equivalente":int(near.index.duplicated(keep=False).sum()),
 "predios_por_codigo":codes.value_counts().to_dict()}
display(pd.Series(soporte).to_frame("valor"))
assert not no_funcion
print("La asignación por vecino no verifica la microzona real de cada predio. Conservar distancia y origen del código.")

## 4b. Correspondencia entre predios oficiales y exposición GEM

El extracto GEM contiene coordenadas redondeadas a cuatro decimales. Se cuenta la ambigüedad antes de
aceptar atributos prediales. Estar cerca no demuestra ser el mismo edificio. Un lote oficial sigue siendo lote.


In [ ]:
ex=pd.read_csv(ROOT/"data/externos/gem_exposicion.csv",dtype={"FLOORS":str})
exg=gpd.GeoDataFrame(ex,geometry=gpd.points_from_xy(ex.lon,ex.lat),crs=4326).to_crs(CRS_METROS)
match=gpd.sjoin_nearest(metric,exg[["id","geometry"]],how="left",max_distance=15,distance_col="dist_gem_m")
num=match.groupby(level=0).id.count().reindex(metric.index,fill_value=0)
ids_primera=match.groupby(level=0).id.first().reindex(metric.index)
pisos_oficiales=pd.to_numeric(predios.NPISOS,errors="coerce")
match_stats={"sin_candidato_mas_cercano_hasta_15m":int(num.eq(0).sum()),
 "con_candidato_unico_mas_cercano":int(num.eq(1).sum()),
 "con_empate_entre_candidatos_mas_cercanos":int(num.gt(1).sum()),
 "filas_en_empates":int(match.loc[match.index.isin(num[num.gt(1)].index)].shape[0]),
 "ids_gem_asignados_a_multiples_predios_si_se_toma_primero":int(ids_primera.value_counts().gt(1).sum()),
 "puntos_gem_repetidos_adicionales":int(ex.duplicated(["lon","lat"]).sum()),
 "lotes_oficiales_con_candidato_gem":int((pisos_oficiales.eq(0)&num.gt(0)).sum())}
display(pd.Series(match_stats).to_frame("valor"))
assert int(num.eq(0).sum()+num.eq(1).sum()+num.gt(1).sum())==len(predios)
# Comparación de catálogos: coincidencia por ID, no suma de archivos.
old=json.loads((ROOT/"data/externos/usgs_sismos_2026-08.geojson").read_text())
new=json.loads((ROOT/"data/externos/usgs_replicas.geojson").read_text())
a={f["id"] for f in old["features"]};b={f["id"] for f in new["features"]}
catalogos={"extracto_anterior":len(a),"extracto_actualizado":len(b),"comunes":len(a&b),"union_eventos":len(a|b)}
display(pd.Series(catalogos).to_frame("eventos"))
guardar_resumen("99_cruces_adicionales",{"gem_match":match_stats,"catalogos":catalogos})

## 5. Selección decidida para la siguiente capa de plata

La base principal es el inventario del evento. Se eligen **dos familias externas** para el análisis local:
OSM para servicios/vías y GEM para contexto de suelo. Comunas y barrios apoyan las uniones.
USGS aporta ahora ShakeMap local y observaciones de entrada, además de los catálogos. Se conserva como sacudida/contexto y apoyo al contraste; no justifica por sí solo que suelo o altura expliquen el daño.

In [ ]:
decisiones=[
 ("Predios oficiales","Base principal","Estado satelital, ubicación e identificación","Llave OBJECTID; NPISOS=0 lote, NULL desconocido; promedios solo positivos; revisar contornos repetidos"),
 ("OSM comunas","Auxiliar territorial","Delimitar agrupaciones y completar código faltante","Control de cero/múltiples coincidencias y discrepancias con código original"),
 ("OSM barrios","Auxiliar condicionado","Nombre/ubicación territorial","Revisar cobertura; no unir BARRIO catastral con osm_id ni tomar primera coincidencia sin control"),
 ("OSM salud","Seleccionada: servicios","Proximidad a atención","Separar hospital/clínica y urgencias sí/no/sin dato; capacidad y operatividad desconocidas"),
 ("OSM emergencia","Seleccionada: servicios","Proximidad a bomberos","Separar 11 bomberos de 52 policías; no inferir vehículos o personal"),
 ("OSM vías principales","Seleccionada: servicios","Proximidad e interferencia potencial","Metros EPSG:9377; conservar restricciones; no usar para tiempos de ruta"),
 ("OSM educación","Contexto, fuera del cálculo de albergues","Exposición educativa","No declarar albergues sin habilitación/capacidad"),
 ("OSM puntos de encuentro","Sin registros utilizables","Cobertura vacía","Tabla opcional con esquema; no interpretar como inexistencia real"),
 ("USGS catálogo","Contexto de eventos","Magnitud, tiempo, profundidad","Conservar magType y profundidad Z; no llamar réplicas a todo ni usar MMI como intensidad local"),
 ("GEM grilla","Seleccionada: suelo","Asignación modelada de sitio","ampcode y distancia; Vs30 de referencia; no imputar noaf por falta de cruce"),
 ("GEM amplificación","Seleccionada: suelo","Respuesta espectral modelada","Llave código-nivel-período; factores y sigmas separados; declarar supuestos de excitación"),
 ("PBF Colombia","Respaldo de bronce","Reproducir extracción OSM","No duplicarlo como fuente de evidencia ni cargarlo completo sin necesidad"),
 ("GEM exposición recortada","Seleccionada agregada; cruce predial condicionado","Tipología, ocupación nocturna y contexto constructivo","Recuperar coordenadas sin redondear; revisar empate/reutilización; jamás imputar pisos a lotes"),
 ("GEM licuación/ladera","Contexto de sitio condicionado","Susceptibilidad modelada y pendiente","Conservar categorías, metadatos de unidad y distancia de asignación; no equivale a afectación"),
 ("USGS ShakeMap Cali","Seleccionada: sacudida modelada","MMI, PGA, PGV, PSA locales del evento","Preservar XML, versión y unidades; distancia al nodo; evitar doble conteo del sitio"),
 ("USGS stationlist Cali","Apoyo a consistencia interna","1 registro instrumental y 32 macroseísmicos","No son 33 acelerógrafos; comprobar independencia antes de llamarlo validación externa"),
 ("USGS eventos actualizados","Contexto; consolidar con extracto anterior","Catálogo de eventos a radio/ventana diferente","Deduplicar por evento_id; no sumar ambos extractos como sismos distintos")]
seleccion=pd.DataFrame(decisiones,columns=["dataset","decision","utilidad","reglas_plata"])
display(HTML(seleccion.to_html(index=False)))
guardar_resumen("99_seleccion_para_plata",{"datasets":17,"criterios":"Guía del evento, fuentes materializadas, calidad y cobertura",
 "territorio":territorio,"gem_match":match_stats,"catalogos":catalogos,"salud":salud_stats,"suelo":soporte,"seleccion":seleccion.to_dict("records")})

## 6. Fuentes investigadas que todavía no son bronce local

No se simula un EDA de archivos que no están descargados. La siguiente tabla es una decisión de
adquisición y alcance, no un perfil estadístico. Las referencias y enlaces están en `docs/FUENTES.md`.

In [ ]:
pendientes=pd.DataFrame([
 ("C01","ComCat/FDSN","Parcial: existe solo extracto regional","Más eventos/ciudades requieren otro extracto y homologar magType"),
 ("C02","ShakeMap","Materializada para Cali: notebook 15","Falta extracto comparable de otras ciudades/eventos, con versión y unidades"),
 ("C03","DYFI","Parcial: 32 observaciones macroseísmicas en notebook 16","El extracto stationlist no preserva nresp ni incertidumbre; recuperar producto DYFI original"),
 ("C04","PAGER","Contexto modelado","No reemplaza víctimas observadas ni datos por comuna"),
 ("C05","Vs30 global","Reserva metodológica","GEM local ya materializado; usar si falta otra ciudad o para sensibilidad"),
 ("C06","SGC instrumental/histórico","Antecedentes para contraste nacional","Faltan fichas/exportación de efectos por sitio y eventos comparables"),
 ("C07","Portal SGC","Descubrimiento","Seleccionar capa real antes de perfilar"),
 ("C08","Microzonificación original","Referencia técnica","Original con acceso limitado; GEM es el insumo local disponible"),
 ("C09","GEM exposición/riesgo","Exposición y sitio de Cali materializados: notebooks 13–14","Quito/Santiago y escenarios no materializados en este corte; conservar condición de modelo"),
 ("C10","NSR-10","Referencia técnica","No es un inventario de daños o cumplimiento"),
 ("C11","DANE población","Prioridad para exposición","Falta tabla temática y unión; no estimar habitantes solo contando la muestra de edificios"),
 ("C12","UNGRD","Histórico observado complementario","API sin snapshot local; homologar períodos, municipios y ayuda; no confundir filas con sismos"),
 ("C13","Copernicus EMSR916","Validación satelital potencial","Faltan productos y áreas de interés verificadas"),
 ("C14","ARIA","Validación satelital potencial","Falta producto específico y evaluación de independencia del inventario del evento"),
 ("C15","ReliefWeb/OPS/PNUD","Validación de balances","Faltan tablas locales con corte y unidad geográfica comparable"),
 ("C16","HDX histórico","Prioridad histórica después del núcleo","Falta inspeccionar XLSX y solape con UNGRD; el nombre 2016–2026 no acredita cobertura"),
 ("C17","OSM/Geofabrik","Materializada: siete capas y PBF","EDA en notebooks 02–08 y 12"),
 ("C18","Alcaldía de Cali","Pendiente de acceso y recurso específico","No atribuir ausencia real a un bloqueo web"),
 ("C19","FUNVISIS","Pendiente para comparación venezolana","Faltan catálogo/fichas y balances verificables"),
 ("C20","Inventario del evento","Materializada y seleccionada","EDA en notebook 01; interpretación satelital y restricciones de redistribución")
],columns=["id_catalogo","fuente","decision_de_alcance","requisito"])
display(HTML(pendientes.to_html(index=False)))

## 7. Contrato propuesto para plata y límites de las decisiones

Tablas mínimas:

- `silver_<equipo>_predios`: identidad de fila, versiones originales y tipadas, estado satelital,
  calidad de geometría, código territorial original/asignado, `sin_edificacion_registrada` para NPISOS=0, y método/bandera solo cuando se estimen NULL justificados. Nunca imputar los lotes.
- `silver_<equipo>_territorios`: tipo, código de análisis, `osm_id`, nombre, geometría y procedencia.
- `silver_<equipo>_servicios`: `osm_id`, tipo, urgencias sí/no/sin dato, operatividad y capacidad
  como campos faltantes cuando no existen; sin equivalencias automáticas entre policía, bomberos y salud.
- `silver_<equipo>_vias`: tramo, jerarquía, longitud, restricciones y procedencia.
- `silver_<equipo>_sitio_grilla` y `silver_<equipo>_amplificacion`: coordenadas/código de sitio y
  funciones con llave `(ampcode, nivel_roca_g, periodo_s)`; metadatos de referencia conservados.
- `silver_<equipo>_eventos`: ID de evento, fechas UTC, magnitud/tipo, profundidad km y consulta de origen.

Puertas de calidad antes de publicar plata: llaves no nulas y únicas según la unidad declarada;
conservar número de predios o justificar cuarentena; cero geometrías vacías en cruces; controlar
múltiples coincidencias; no perder filas por una unión uno-a-muchos; guardar distancias en metros;
distinguir faltantes de cero; promediar pisos solo en construcciones registradas y evaluar cualquier estimación de NULL; conservar SHA-256 y corte por fuente.

**Límites:** este trabajo elige fuentes y reglas, pero no migra el pipeline existente a ese contrato.
No ajusta pesos desde 19 antecedentes seleccionados ni prueba causalidad del suelo. La regla de lotes definida por el dueño queda en el contrato de plata. La actualización remota 8418820, integrada al publicar, ya incorpora sin_construccion_registrada y evita imputar esos predios; también hace deterministas los empates GEM. Esto no acredita por sí solo identidad predial ni elimina las restricciones de correspondencia medidas en el EDA. El inventario
cubre una muestra de Cali; faltan datos de capacidad real, tiempos y recursos disponibles para
optimizar una asignación. Para comparar con otras ciudades faltan los datasets comparables materializados.